In [1]:
!pip install -q ultralytics
from pathlib import Path
import zipfile
import shutil
import cv2
import matplotlib.pyplot as plt

from ultralytics import YOLO
from google.colab import drive

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
ZIP_PATH = "/content/drive/MyDrive/Smart_Warehouse_Inventory_ Project.zip"

EXTRACT_PATH = "/content"

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Project extracted successfully!")

Project extracted successfully!


In [8]:
import shutil
from pathlib import Path

SOURCE = Path("/content/drive/MyDrive/YOLO_Training/warehouse_inventory/weights/best.pt")

DESTINATION = Path("/content/Smart_Warehouse_Inventory_ Project/models")

DESTINATION.mkdir(exist_ok=True)

shutil.copy(SOURCE, DESTINATION)

print("best.pt copied successfully!")

best.pt copied successfully!


In [9]:
PROJECT_PATH = Path("/content/Smart_Warehouse_Inventory_ Project")

MODEL_PATH = PROJECT_PATH / "models" / "best.pt"

TEST_IMAGES = PROJECT_PATH / "test_images"

OUTPUT_FOLDER = PROJECT_PATH / "output" / "prediction"

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print(MODEL_PATH.exists())
print(TEST_IMAGES.exists())
print(OUTPUT_FOLDER.exists())

True
True
True


In [10]:
model = YOLO(str(MODEL_PATH))

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [21]:
# Read All Images
TEST_IMAGES = Path("/content/drive/MyDrive/test_images")

image_paths = sorted(TEST_IMAGES.glob("*"))

print(f"Total Images : {len(image_paths)}")

for image in image_paths:
    print(image.name)

Total Images : 5
warehouse01.jpg
warehouse02.jpg
warehouse03.jpg
warehouse04.jpg
warehouse05.jpg


In [22]:
# Create Results folder
from pathlib import Path

RESULTS_FOLDER = Path("/content/drive/MyDrive/YOLO_Training/real_world_testing")

RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

print("Results Folder Created!")

Results Folder Created!


In [23]:
# predict and save images
import cv2

for image_path in image_paths:

    results = model.predict(
        source=str(image_path),
        imgsz=1280,
        conf=0.10,
        save=False,
        verbose=False
    )

    result = results[0]

    annotated_image = result.plot()

    save_path = RESULTS_FOLDER / f"{image_path.stem}_prediction.jpg"

    cv2.imwrite(str(save_path), annotated_image)

print("All Predictions Saved Successfully!")

All Predictions Saved Successfully!


In [24]:
# Verify Predicted images
prediction_images = sorted(RESULTS_FOLDER.glob("*"))

print(f"Total Prediction Images : {len(prediction_images)}")

for image in prediction_images:
    print(image.name)

Total Prediction Images : 5
warehouse01_prediction.jpg
warehouse02_prediction.jpg
warehouse03_prediction.jpg
warehouse04_prediction.jpg
warehouse05_prediction.jpg


In [25]:
# Compare Original VS Prediction
import matplotlib.pyplot as plt
import cv2

for original_path in image_paths:

    prediction_path = RESULTS_FOLDER / f"{original_path.stem}_prediction.jpg"

    original = cv2.imread(str(original_path))
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

    prediction = cv2.imread(str(prediction_path))
    prediction = cv2.cvtColor(prediction, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(16,7))

    plt.subplot(1,2,1)
    plt.imshow(original)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(prediction)
    plt.title("YOLO Prediction")
    plt.axis("off")

    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [26]:
# Prediction Summary
for image_path in image_paths:

    results = model.predict(
        source=str(image_path),
        imgsz=1280,
        conf=0.10,
        save=False,
        verbose=False
    )

    result = results[0]

    print("="*60)
    print(f"Image : {image_path.name}")
    print(f"Objects Detected : {len(result.boxes)}")

    for box in result.boxes:

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        print(f"  • {model.names[class_id]} ({confidence:.2f})")

Image : warehouse01.jpg
Objects Detected : 7
  • pallet (0.68)
  • wall (0.23)
  • pillar (0.17)
  • bracket (0.12)
  • bracket (0.12)
  • rack (0.11)
  • bracket (0.10)
Image : warehouse02.jpg
Objects Detected : 1
  • rack (0.48)
Image : warehouse03.jpg
Objects Detected : 1
  • floor_decal (0.16)
Image : warehouse04.jpg
Objects Detected : 0
Image : warehouse05.jpg
Objects Detected : 0


In [ ]:
# Observations

### - Tested the trained YOLOv8 model on five completely unseen warehouse images downloaded from Pexels.
### - Initially, using the default inference settings (`imgsz=640`, `conf=0.25`) resulted in few or no detections.
### - Increasing the inference image size to `1280` and reducing the confidence threshold to `0.10` improved detection performance on external images.
### - The model successfully detected objects such as pallets, walls, pillars, racks, floor decals, and brackets in several unseen images.
### - Some images still produced no detections because they differed significantly from the training dataset in terms of camera angle, object scale, lighting, and warehouse layout (domain shift).
### - This experiment demonstrates that model performance depends not only on training accuracy but also on how closely real-world data matches the training distribution.